<a href="https://colab.research.google.com/github/maricmello/PLN_BERT/blob/main/Analise_Sentimento_BERTimbau.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise de sentimento em português com BERTimbau


Baseado no notebook da aula `05_BERT.ipynb` (BERT no IMDB), adaptado para classificar resenhas em português como positivas ou negativas.

O objetivo é ensinar um modelo a dizer se uma resenha escrita em português é positiva ou negativa. Partimos do notebook da aula, que fazia o fine-tuning do BERT em resenhas do IMDB, e mantivemos a mesma estrutura: o Dataset, a classe `BertClassifier`, o laço de treino com acúmulo de gradiente e o teste de overfit em um batch.

As diferenças vêm, em boa parte, do próprio dado. Usamos o BERTimbau, que foi pré-treinado em português, e o dataset `jvanz/portuguese_sentiment_analysis`, que junta resenhas de filmes, aplicativos, jogos e produtos. As resenhas são bem mais curtas que as do IMDB, então o `max_length` é de 144 tokens em vez de 512. As classes também são desbalanceadas, com cerca de 81% de positivas. Para lidar com isso, o treino usa pesos de classe na função de perda, o melhor modelo é escolhido pelo F1 macro e o limiar de decisão é ajustado na validação. Por fim, comparamos o resultado com baselines simples, calculamos intervalos de confiança e um teste de significância e analisamos os erros.

## Instruções

Treinar e medir o desempenho do BERTimbau (`neuralmind/bert-base-portuguese-cased`) para classificação binária no dataset `jvanz/portuguese_sentiment_analysis` (20k/5k/5k amostras de treino/validação/teste).

Assim como na aula:
- Implementar o próprio laço de treinamento.
- Verificar overfit em um batch.
- Implementar o acúmulo de gradiente.

Para rodar: Ambiente de execução → Alterar o tipo de ambiente de execução → GPU, e depois Executar tudo.

# Instalação e versões

In [1]:
!pip install -q datasets

In [2]:
import importlib.metadata as md_versoes
import platform

versoes = {'python': platform.python_version()}
for pacote in ['torch', 'transformers', 'datasets', 'scikit-learn', 'scipy', 'pandas', 'numpy']:
    try:
        versoes[pacote] = md_versoes.version(pacote)
    except md_versoes.PackageNotFoundError:
        versoes[pacote] = '-'
print(versoes)

{'python': '3.13.16', 'torch': '2.14.1+cu130', 'transformers': '5.22.1', 'datasets': '4.8.5', 'scikit-learn': '1.6.1', 'scipy': '1.16.3', 'pandas': '2.2.3', 'numpy': '2.4.6'}


# Fixando a seed

In [ ]:
import json
import math
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from scipy.stats import binomtest
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, BertModel, get_linear_schedule_with_warmup

In [ ]:
if torch.cuda.is_available():
   dev = "cuda:0"
   print(torch.cuda.get_device_name(dev))
else:
   dev = "cpu"
print(dev)
device = torch.device(dev)
nome_gpu = torch.cuda.get_device_name(dev) if device.type == 'cuda' else 'CPU'

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## Pasta para salvar os resultados

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = '/content/drive/MyDrive/sentimento_bertimbau'
except ImportError:
    PASTA = './sentimento_bertimbau'

os.makedirs(PASTA, exist_ok=True)
with open(os.path.join(PASTA, 'versoes.txt'), 'w') as f:
    for pacote, versao in versoes.items():
        f.write(f'{pacote}=={versao}\n')
print('Pasta:', PASTA)

## Preparando Dados

Primeiro, carregamos o dataset do Hugging Face:

In [ ]:
raw = load_dataset('jvanz/portuguese_sentiment_analysis')
print(raw)

## Carregando o dataset

Diferente do IMDB da aula, aqui o dataset já vem com treino, validação e teste. Sorteamos 20k exemplos de treino, 5k de validação e 5k de teste.

Para cada divisão, embaralhamos o split original com a seed e sorteamos um pouco mais de exemplos do que o necessário, já que a limpeza descarta parte deles. Removemos espaços nas pontas, textos vazios e textos duplicados. Depois tiramos da validação qualquer texto que já estivesse no treino, e do teste qualquer texto que estivesse no treino ou na validação. Sem esse cuidado, o modelo seria avaliado em parte com frases que já viu, e o resultado sairia inflado. Os dois `assert` no fim da célula conferem que nenhum texto ficou em mais de um conjunto.

In [ ]:
max_train = 20000
max_valid = 5000
max_test = 5000


def pegar_amostra(split, size, excluir=frozenset()):
    embaralhado = split.shuffle(seed=SEED)
    for fator in (1.5, 3, 6, 12):
        total = min(int(size * fator), len(embaralhado))
        sorteados = embaralhado.select(range(total))
        frame = pd.DataFrame({
            'texto': sorteados['review_text'],
            'rotulo': [int(p) for p in sorteados['polarity']],
        })
        frame['texto'] = frame['texto'].fillna('').astype(str).str.strip()
        frame = frame[frame['texto'].str.len() > 0]
        frame = frame.drop_duplicates('texto')
        frame = frame[~frame['texto'].isin(excluir)]
        if len(frame) >= size or total == len(embaralhado):
            break
    assert len(frame) >= size, 'amostra ficou pequena depois da limpeza'
    return frame.head(size).reset_index(drop=True)


treino = pegar_amostra(raw['train'], max_train)
validacao = pegar_amostra(raw['validation'], max_valid, excluir=set(treino['texto']))
teste = pegar_amostra(raw['test'], max_test, excluir=set(treino['texto']) | set(validacao['texto']))

assert not set(treino['texto']) & set(validacao['texto'])
assert not (set(treino['texto']) | set(validacao['texto'])) & set(teste['texto'])

x_train, y_train = treino['texto'].tolist(), treino['rotulo'].tolist()
x_valid, y_valid = validacao['texto'].tolist(), validacao['rotulo'].tolist()
x_test, y_test = teste['texto'].tolist(), teste['rotulo'].tolist()

print(len(x_train), 'amostras de treino.')
print(len(x_valid), 'amostras de desenvolvimento.')
print(len(x_test), 'amostras de teste.')

for nome, frame in [('treino', treino), ('validação', validacao), ('teste', teste)]:
    positivos = int(frame['rotulo'].sum())
    print(f'{nome}: {positivos} positivos ({positivos / len(frame):.1%}), {len(frame) - positivos} negativos')

print('\n3 primeiras amostras treino:')
for x, y in zip(x_train[:3], y_train[:3]):
    print(y, x[:100])

print('\n3 primeiras amostras validação:')
for x, y in zip(x_valid[:3], y_valid[:3]):
    print(y, x[:100])

No IMDB da aula as classes eram meio a meio. Aqui não: 81,5% das resenhas de treino são positivas, e a proporção é praticamente a mesma na validação (81,4%) e no teste (81,7%). Isso muda a leitura da acurácia. Um modelo que respondesse sempre "positivo", sem aprender nada, já acertaria 81,7% do teste. Por isso usamos também o F1 macro, que é a média do F1 das duas classes e só fica alto se o modelo também reconhecer bem as negativas.

## Carregando BERTimbau e visualizando as transformações

In [ ]:
model_name = 'neuralmind/bert-base-portuguese-cased'

tokenizer = AutoTokenizer.from_pretrained(model_name)
encoded_input = tokenizer("Olá, eu sou uma frase só!")
print(encoded_input)
print("Decoding sentence: ", tokenizer.decode(encoded_input["input_ids"]))

Na aula usamos `padding="max_length"` com 512 tokens. As resenhas aqui são bem mais curtas que as do IMDB, então medimos o tamanho em tokens e usamos o percentil 95 como `max_length`.

O histograma mostra que a maioria das resenhas é curta, com mediana de 22 tokens, mas tem uma cauda longa: o percentil 95 fica em 138 tokens e o texto mais longo da amostra chega a 1.189. Arredondando o percentil 95 para múltiplo de 16 chegamos a `max_length = 144`. Com esse valor, só 4,6% dos textos são cortados, e cada exemplo ocupa menos de um terço do espaço que ocuparia com os 512 tokens da aula, o que deixa o treino bem mais leve. O gráfico ao lado confirma o desbalanceamento entre as classes.

In [ ]:
amostra = treino.sample(min(2000, len(treino)), random_state=SEED)
comprimentos = np.array([len(ids) for ids in tokenizer(amostra['texto'].tolist(), truncation=False)['input_ids']])

p95 = int(np.percentile(comprimentos, 95))
max_length = int(min(512, int(np.ceil(p95 / 16) * 16)))
fracao_cortada = float((comprimentos > max_length).mean())

print('Tokens por texto - mediana:', int(np.median(comprimentos)), '| p95:', p95, '| máximo:', int(comprimentos.max()))
print(f'max_length: {max_length} ({fracao_cortada:.1%} dos textos são cortados)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(np.clip(comprimentos, 0, 300), bins=40, color='#0072B2')
axes[0].axvline(max_length, color='black', linestyle='--', label=f'max_length = {max_length}')
axes[0].set_xlabel('Tokens por texto (cortado em 300)')
axes[0].set_ylabel('Textos')
axes[0].legend()

contagem = treino['rotulo'].map({0: 'negativo', 1: 'positivo'}).value_counts().reindex(['negativo', 'positivo'])
axes[1].bar(contagem.index, contagem.values, color=['#E69F00', '#0072B2'])
axes[1].set_ylabel('Textos no treino')
axes[1].set_title('Distribuição das classes')
plt.tight_layout()
plt.savefig(os.path.join(PASTA, 'distribuicao_dos_dados.png'), dpi=150)
plt.show()

## Definindo Dataset e Dataloader

In [ ]:
class ResenhasDataset(torch.utils.data.Dataset):
    def __init__(self, sentences,
                 labels,
                 model_tokens=model_name,
                 max_length=max_length):
        super().__init__()

        self.tokenizer = AutoTokenizer.from_pretrained(model_tokens)

        encoded = self.tokenizer(sentences,
                                 padding="max_length",
                                 truncation=True,
                                 max_length=max_length)
        self.tokens = encoded["input_ids"]
        self.mask_attention = encoded["attention_mask"]
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
      return torch.tensor(self.tokens[idx]).long(),\
             torch.tensor(self.mask_attention[idx]).long(),\
             torch.tensor(self.labels[idx]).long()

## Modelo de classificação de reviews

A arquitetura é a mesma da aula. O `BertModel` do BERTimbau processa o texto, e usamos o `pooler_output`, um vetor de 768 dimensões gerado a partir do token `[CLS]` que resume a frase inteira. Em cima dele aplicamos um dropout de 0,1 e uma camada linear que leva de 768 para 2 saídas, uma para cada classe. Só essa camada linear começa com pesos aleatórios. O resto da rede parte do pré-treino e é ajustado junto durante o fine-tuning. Por isso, ao carregar o modelo, a biblioteca avisa que as camadas `cls.*` do pré-treino foram ignoradas: elas serviam para adivinhar palavras escondidas e não são usadas na classificação.

In [ ]:
class BertClassifier(nn.Module):
    def __init__(self, num_class, model_name=model_name, dropout=0.1):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_class)

    def forward(self, input_ids, attention_mask):
        pooled = self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        return self.classifier(self.dropout(pooled))

## Laço de Treinamento

O laço segue o da aula. O acúmulo de gradiente soma os gradientes de 2 mini-batches antes de atualizar os pesos, o que dá o efeito de um batch maior sem precisar de mais memória na GPU. Cada loss é dividida pelo tamanho do grupo para que a soma equivalha à média, e o último grupo da época, que pode ter menos mini-batches, é tratado à parte.

O otimizador é o AdamW com weight decay de 0,01, aplicado a todos os pesos menos os bias e as camadas de LayerNorm, como é comum no fine-tuning de BERT. A taxa de aprendizado sobe aos poucos nos primeiros 10% dos passos (o warmup) e depois cai em linha reta até zero. Isso evita que as primeiras atualizações, ainda com a camada de classificação aleatória, estraguem o que o modelo aprendeu no pré-treino. Antes de cada passo do otimizador, o clipping limita a norma do gradiente a 1, para segurar atualizações grandes demais. A função de perda é a entropia cruzada, com um peso para cada classe, definido mais adiante. Como a T4 não suporta `bfloat16`, o treino roda em `float32`.

In [ ]:
use_bf16 = device.type == 'cuda' and torch.cuda.get_device_capability(device)[0] >= 8
print('Usando bfloat16:', use_bf16)


def forward_batch(model, batch, weight=None):
    input_ids, attention_mask, labels = (tensor.to(device) for tensor in batch)
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
        logits = model(input_ids, attention_mask)
    loss = F.cross_entropy(logits.float(), labels, weight=weight)
    return loss, logits, labels


def build_optimizer_and_scheduler(model, lr, loader, accum_steps, max_epochs, weight_decay=0.01, warmup_ratio=0.1):
    no_decay = ['bias', 'LayerNorm.weight']
    grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            'weight_decay': weight_decay,
        },
        {
            'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0,
        },
    ]
    optimizer = torch.optim.AdamW(grouped_parameters, lr=lr)
    total_steps = math.ceil(len(loader) / accum_steps) * max_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(warmup_ratio * total_steps), num_training_steps=total_steps
    )
    return optimizer, scheduler


def train_epoch(model, loader, optimizer, scheduler, accum_steps, max_grad_norm=1.0, weight=None):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    num_batches = len(loader)
    remainder = num_batches % accum_steps or accum_steps
    last_group_start = num_batches - remainder

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    progress_bar = tqdm(loader, leave=False)
    for step, batch in enumerate(progress_bar):
        group_size = accum_steps if step < last_group_start else remainder

        loss, logits, labels = forward_batch(model, batch, weight)
        (loss / group_size).backward()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=-1) == labels).sum().item()
        total_samples += labels.size(0)

        if (step + 1) % accum_steps == 0 or (step + 1) == num_batches:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            progress_bar.set_postfix(loss=total_loss / total_samples, acc=total_correct / total_samples)

    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def evaluate(model, loader, weight=None):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in tqdm(loader, leave=False):
        loss, logits, labels = forward_batch(model, batch, weight)
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=-1) == labels).sum().item()
        total_samples += labels.size(0)

    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    probs = []
    for batch in tqdm(loader, leave=False):
        _, logits, _ = forward_batch(model, batch)
        probs.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(probs)

## Overfitting

Verifique o overfit em um batch

In [ ]:
lr = 1e-4
bs = 5
bs_acc_step = 2
max_epochs = 7
num_class = 2

params = {
    'lr': lr,
    'bs': bs,
    'max_epochs': max_epochs,
    'num_class': num_class,
    'bs_acc_step': bs_acc_step
}

In [ ]:
train_dataset = ResenhasDataset(x_train[:100], y_train[:100])

loader_train = DataLoader(train_dataset, batch_size=params['bs'], shuffle=True)

In [ ]:
x_train_, x_train_mask, y_train_ = next(iter(DataLoader(train_dataset, batch_size=5)))
print("\nDimensões dos dados de um minibatch:", x_train_.size())
print("\nDimensões dos dados de um minibatch:", y_train_.size())

In [ ]:
overfit_model = BertClassifier(params['num_class']).to(device)
overfit_optimizer = torch.optim.AdamW(overfit_model.parameters(), lr=params['lr'])
overfit_batch = (x_train_, x_train_mask, y_train_)

overfit_model.train()
for step in range(1, 51):
    overfit_optimizer.zero_grad(set_to_none=True)
    loss, logits, labels = forward_batch(overfit_model, overfit_batch)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(overfit_model.parameters(), 1.0)
    overfit_optimizer.step()

    accuracy = (logits.argmax(dim=-1) == labels).float().mean().item()
    if step % 5 == 0:
        print(f'Passo {step:02d} - loss: {loss.item():.4f} | acurácia: {accuracy:.2f}')

overfit_loss_final, overfit_acc_final = loss.item(), accuracy

del overfit_model, overfit_optimizer
if device.type == 'cuda':
    torch.cuda.empty_cache()

O teste passou. Com 5 resenhas e taxa de aprendizado de 1e-4, a loss caiu de 0,18 no passo 5 para 0,0065 no passo 10 e chegou a praticamente zero no passo 50, sempre com acurácia 1,00. Na aula, com o BERT no IMDB, a loss tinha ido de 0,26 a 0,0001 no mesmo número de passos. Se o modelo consegue decorar um batch pequeno, o modelo, a loss e o laço estão ligados do jeito certo, e podemos seguir para o treino completo.

## Treinamento - BERTimbau

Em relação à aula, a taxa de aprendizado é menor, 3e-5 em vez de 1e-4, um valor mais usual para fine-tuning de BERT, e o treino tem 3 épocas em vez de 5. Como as resenhas são curtas e o `max_length` é 144, cabem batches de 16 exemplos. Com 2 passos de acúmulo, o batch efetivo fica em 32, contra 20 na aula (10 × 2). No total são 1.875 passos do otimizador.

Três escolhas não aparecem nos hiperparâmetros. A loss usa pesos por classe, explicados logo abaixo. A melhor época é escolhida pelo F1 macro da validação, e não pela acurácia, também por causa do desbalanceamento. E os loaders de validação e teste não embaralham os dados, para que as probabilidades saiam na mesma ordem dos textos e possam ser usadas na análise de erros.

In [ ]:
lr = 3e-5
bs = 16
bs_acc_step = 2
max_epochs = 3
num_class = 2

params = {
    'lr': lr,
    'bs': bs,
    'max_epochs': max_epochs,
    'num_class': num_class,
    'bs_acc_step': bs_acc_step
}
print('batch efetivo:', params['bs'] * params['bs_acc_step'])

### Peso das classes

Com 81,5% de positivas no treino, os erros nas resenhas negativas têm pouco efeito na loss média, e o modelo tende a favorecer a classe majoritária. Para compensar, cada classe recebe um peso inversamente proporcional à sua frequência no treino, calculado como o total de exemplos dividido por duas vezes o número de exemplos da classe. Com 3.698 negativas e 16.302 positivas, o peso das negativas fica em 2,70 e o das positivas em 0,61. Na prática, errar uma negativa custa cerca de 4,4 vezes mais do que errar uma positiva.

A loss de validação usa os mesmos pesos, para que as curvas de treino e de validação fiquem na mesma escala. A acurácia e o F1 não dependem dos pesos.

In [ ]:
contagem_classes = np.bincount(y_train, minlength=2)
pesos_classes = torch.tensor(len(y_train) / (2 * contagem_classes), dtype=torch.float32).to(device)

print('exemplos por classe (negativo, positivo):', contagem_classes.tolist())
print('pesos por classe (negativo, positivo):', [round(p, 4) for p in pesos_classes.tolist()])

In [ ]:
train_dataset = ResenhasDataset(x_train, y_train)
val_dataset = ResenhasDataset(x_valid, y_valid)
test_dataset = ResenhasDataset(x_test, y_test)

loader_train = DataLoader(train_dataset, batch_size=params['bs'], shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
loader_val = DataLoader(val_dataset, batch_size=params['bs'] * 2, shuffle=False)
loader_test = DataLoader(test_dataset, batch_size=params['bs'] * 2, shuffle=False)

In [ ]:
x_train_, x_train_mask, y_train_ = next(iter(DataLoader(train_dataset, batch_size=5)))
print("\nDimensões dos dados de um minibatch:", x_train_.size())
print("\nDimensões dos dados de um minibatch:", y_train_.size())

In [ ]:
CAMINHO_MODELO = os.path.join(PASTA, 'best_bertimbau.pt')

model = BertClassifier(params['num_class']).to(device)
optimizer, scheduler = build_optimizer_and_scheduler(
    model, params['lr'], loader_train, params['bs_acc_step'], params['max_epochs']
)

best_val_f1 = -1.0
best_epoch = None
history = []
inicio = time.time()

for epoch in range(1, params['max_epochs'] + 1):
    train_loss, train_accuracy = train_epoch(
        model, loader_train, optimizer, scheduler, params['bs_acc_step'], weight=pesos_classes
    )
    val_loss, val_accuracy = evaluate(model, loader_val, weight=pesos_classes)
    val_f1 = f1_score(y_valid, (predict_proba(model, loader_val) >= 0.5).astype(int), average='macro')
    history.append({'epoca': epoch, 'train_loss': train_loss, 'train_acc': train_accuracy,
                    'val_loss': val_loss, 'val_acc': val_accuracy, 'val_f1_macro': val_f1})
    print(
        f'Época {epoch} - '
        f'train_loss: {train_loss:.4f} | train_acc: {train_accuracy:.4f} | '
        f'val_loss: {val_loss:.4f} | val_acc: {val_accuracy:.4f} | val_f1_macro: {val_f1:.4f}'
    )

    if val_f1 > best_val_f1:
        best_val_f1, best_epoch = val_f1, epoch
        torch.save(model.state_dict(), CAMINHO_MODELO)

tempo_treino_min = (time.time() - inicio) / 60
print(f'Melhor F1 macro de validação: {best_val_f1:.4f} (época {best_epoch})')
print(f'Tempo de treino: {tempo_treino_min:.1f} min ({nome_gpu})')

with open(os.path.join(PASTA, 'historico_treino.json'), 'w') as f:
    json.dump(history, f, indent=2)

In [ ]:
historico = pd.DataFrame(history)
print(historico.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(historico['epoca'], historico['train_loss'], marker='o', label='treino')
axes[0].plot(historico['epoca'], historico['val_loss'], marker='o', label='validação')
axes[0].set_title('Loss')
axes[0].set_xlabel('Época')
axes[0].legend()
axes[1].plot(historico['epoca'], historico['train_acc'], marker='o', label='acurácia (treino)')
axes[1].plot(historico['epoca'], historico['val_acc'], marker='o', label='acurácia (validação)')
axes[1].plot(historico['epoca'], historico['val_f1_macro'], marker='s', linestyle='--', label='F1 macro (validação)')
axes[1].axvline(best_epoch, color='grey', linestyle=':', label='modelo escolhido')
axes[1].set_title('Métricas')
axes[1].set_xlabel('Época')
axes[1].legend()
for ax in axes:
    ax.set_xticks(historico['epoca'])
plt.tight_layout()
plt.savefig(os.path.join(PASTA, 'curvas_de_treino.png'), dpi=150)
plt.show()

*(escrever: interpretar as curvas: em que época o modelo começa a decorar o treino e qual época foi salva)*

## Avaliação em teste

In [ ]:
model.load_state_dict(torch.load(CAMINHO_MODELO, map_location=device))
test_loss, test_accuracy = evaluate(model, loader_test)
print(f'Teste - loss: {test_loss:.4f} | acurácia: {test_accuracy:.4f}')

### Ajuste do limiar de decisão

Por padrão, uma resenha é classificada como positiva quando a probabilidade passa de 0,5. Os pesos na loss já compensam boa parte do desbalanceamento, mas não há garantia de que 0,5 seja o melhor ponto de corte para o F1 macro. Por isso testamos limiares de 0,05 a 0,95 e ficamos com o que dá o melhor F1 macro. A escolha é feita na validação, e não no teste, para que o teste continue sendo um conjunto que não influenciou nenhuma decisão.

*(escrever: limiar escolhido e quanto ele mudou o F1 macro na validação)*

In [ ]:
p_val = predict_proba(model, loader_val)

limiares = np.round(np.arange(0.05, 0.96, 0.01), 2)
f1_por_limiar = [f1_score(y_valid, (p_val >= t).astype(int), average='macro') for t in limiares]
melhor_f1_val = max(f1_por_limiar)
empatados = [t for t, f in zip(limiares, f1_por_limiar) if f >= melhor_f1_val - 1e-9]
limiar = float(min(empatados, key=lambda t: abs(t - 0.5)))
print(f'limiar escolhido: {limiar:.2f} | F1 macro validação: {melhor_f1_val:.4f}')

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(limiares, f1_por_limiar, color='#0072B2')
ax.axvline(0.5, color='grey', linestyle=':', label='0,50')
ax.axvline(limiar, color='black', linestyle='--', label=f'escolhido ({limiar:.2f})')
ax.set_xlabel('Limiar')
ax.set_ylabel('F1 macro (validação)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PASTA, 'ajuste_do_limiar.png'), dpi=150)
plt.show()

### Resultado no teste

In [ ]:
ROTULOS = ['negativo', 'positivo']
y_teste = np.array(y_test)

teste['p_pos'] = predict_proba(model, loader_test)
teste['previsto_05'] = (teste['p_pos'] >= 0.5).astype(int)
teste['previsto'] = (teste['p_pos'] >= limiar).astype(int)

for nome, coluna in [('limiar 0,50', 'previsto_05'), (f'limiar {limiar:.2f}', 'previsto')]:
    print(f"BERTimbau, {nome}: acurácia {accuracy_score(y_teste, teste[coluna]):.4f} | "
          f"F1 macro {f1_score(y_teste, teste[coluna], average='macro'):.4f}")

acuracia = accuracy_score(y_teste, teste['previsto'])
f1_macro = f1_score(y_teste, teste['previsto'], average='macro')
print()
print(classification_report(y_teste, teste['previsto'], target_names=ROTULOS, digits=4, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, coluna, titulo in [(axes[0], 'previsto_05', 'Limiar 0,50'), (axes[1], 'previsto', f'Limiar {limiar:.2f}')]:
    ConfusionMatrixDisplay.from_predictions(y_teste, teste[coluna], display_labels=ROTULOS,
                                            cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(titulo)
plt.tight_layout()
plt.savefig(os.path.join(PASTA, 'matriz_de_confusao.png'), dpi=150)
plt.show()

### Comparação com baselines

Usamos três referências. A classe majoritária responde sempre "positivo" e mostra o piso, ou seja, o que se consegue sem aprender nada. O TF-IDF com regressão logística é o baseline clássico de classificação de texto: representa cada resenha pelas palavras e pares de palavras que ela contém e treina em segundos, sem GPU. A terceira é a mesma regressão logística com `class_weight='balanced'`, que dá mais peso à classe negativa. Ela deixa a comparação justa, porque o BERTimbau também foi treinado com pesos por classe.

*(escrever: comparar acurácia, F1 macro e recall das negativas entre os modelos)*

In [ ]:
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
x_tfidf_treino = tfidf.fit_transform(x_train)
x_tfidf_teste = tfidf.transform(x_test)

baselines = {
    'Classe majoritária': DummyClassifier(strategy='most_frequent'),
    'TF-IDF + reg. logística': LogisticRegression(max_iter=1000),
    'TF-IDF + reg. logística (balanceada)': LogisticRegression(max_iter=1000, class_weight='balanced'),
}
previsoes = {}
for nome, clf in baselines.items():
    clf.fit(x_tfidf_treino, y_train)
    previsoes[nome] = clf.predict(x_tfidf_teste)
previsoes['BERTimbau (limiar 0,50)'] = teste['previsto_05'].to_numpy()
previsoes['BERTimbau (limiar ajustado)'] = teste['previsto'].to_numpy()


def recall_negativo(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return float(((p == 0) & (y == 0)).sum() / (y == 0).sum())


comparacao = pd.DataFrame([
    {'modelo': nome,
     'acurácia': accuracy_score(y_teste, p),
     'F1 macro': f1_score(y_teste, p, average='macro'),
     'recall negativo': recall_negativo(y_teste, p)}
    for nome, p in previsoes.items()
])
print(comparacao.round(4).to_string(index=False))
comparacao.round(4).to_csv(os.path.join(PASTA, 'comparacao_modelos.csv'), index=False)

### Intervalos de confiança e teste de McNemar

In [ ]:
n_bootstrap = 1000
rng = np.random.default_rng(SEED)
indices_boot = rng.integers(0, len(y_teste), size=(n_bootstrap, len(y_teste)))


def intervalo_bootstrap(previsto):
    accs = np.array([accuracy_score(y_teste[idx], previsto[idx]) for idx in indices_boot])
    f1s = np.array([f1_score(y_teste[idx], previsto[idx], average='macro') for idx in indices_boot])
    return np.percentile(accs, [2.5, 97.5]), np.percentile(f1s, [2.5, 97.5]), f1s


nome_tfidf = max([m for m in previsoes if m.startswith('TF-IDF')],
                 key=lambda m: f1_score(y_teste, previsoes[m], average='macro'))
intervalos, f1s_boot = {}, {}
for nome in ['BERTimbau (limiar ajustado)', nome_tfidf]:
    ic_acc, ic_f1, f1s_boot[nome] = intervalo_bootstrap(previsoes[nome])
    intervalos[nome] = {'acuracia_ic95': ic_acc.tolist(), 'f1_macro_ic95': ic_f1.tolist()}
    print(f'{nome}: acurácia IC95% [{ic_acc[0]:.4f}, {ic_acc[1]:.4f}] | F1 macro IC95% [{ic_f1[0]:.4f}, {ic_f1[1]:.4f}]')

diferenca = f1s_boot['BERTimbau (limiar ajustado)'] - f1s_boot[nome_tfidf]
ic_diferenca = np.percentile(diferenca, [2.5, 97.5])
print(f'\nganho de F1 macro sobre "{nome_tfidf}": {diferenca.mean():.4f} '
      f'IC95% [{ic_diferenca[0]:.4f}, {ic_diferenca[1]:.4f}]')

acerta_bert = previsoes['BERTimbau (limiar ajustado)'] == y_teste
acerta_tfidf = previsoes[nome_tfidf] == y_teste
so_bert = int((acerta_bert & ~acerta_tfidf).sum())
so_tfidf = int((~acerta_bert & acerta_tfidf).sum())
p_mcnemar = binomtest(so_bert, so_bert + so_tfidf, 0.5).pvalue if so_bert + so_tfidf > 0 else 1.0
print(f'McNemar: só BERTimbau acertou {so_bert} | só TF-IDF acertou {so_tfidf} | p-valor = {p_mcnemar:.2e}')

*(escrever: interpretar os intervalos de confiança e o p-valor do McNemar)*

## Análise de erros

In [ ]:
teste['confianca'] = np.where(teste['previsto'] == 1, teste['p_pos'], 1 - teste['p_pos'])
erros = teste[teste['previsto'] != teste['rotulo']].sort_values('confianca', ascending=False)
erros.to_csv(os.path.join(PASTA, 'erros_do_modelo.csv'), index=False)

print(f'{len(erros)} erros em {len(teste)} exemplos ({len(erros) / len(teste):.1%})')
print(f"negativas previstas como positivas: {int((erros['rotulo'] == 0).sum())}")
print(f"positivas previstas como negativas: {int((erros['rotulo'] == 1).sum())}")

for _, linha in erros.head(10).iterrows():
    print(f"\nreal: {ROTULOS[linha['rotulo']]} | previsto: {ROTULOS[linha['previsto']]} | confiança: {linha['confianca']:.2f}")
    print(linha['texto'][:250])

*(escrever: comentar os erros mais confiantes)*

#### Anotação manual dos erros

Baixar `erros_para_anotar.csv` da pasta do projeto, preencher a coluna `categoria`, salvar como `erros_anotados.csv` na mesma pasta e rodar a célula seguinte.

Categorias: `rotulo_errado`, `ironia`, `negacao`, `misto`, `ambiguo`, `erro_do_modelo`, `outro`.

In [ ]:
n_erros_anotar = 50
amostra_erros = erros.sample(min(n_erros_anotar, len(erros)), random_state=SEED)
para_anotar = pd.DataFrame({
    'texto': amostra_erros['texto'],
    'rotulo_dataset': amostra_erros['rotulo'].map(dict(enumerate(ROTULOS))),
    'previsto_modelo': amostra_erros['previsto'].map(dict(enumerate(ROTULOS))),
    'confianca': amostra_erros['confianca'].round(3),
    'categoria': '',
})
caminho_para_anotar = os.path.join(PASTA, 'erros_para_anotar.csv')
if not os.path.exists(caminho_para_anotar):
    para_anotar.to_csv(caminho_para_anotar, index=False)
    print(len(para_anotar), 'erros salvos em', caminho_para_anotar)
else:
    print(caminho_para_anotar, 'já existe')

In [ ]:
caminho_anotado = os.path.join(PASTA, 'erros_anotados.csv')
categorias = ['rotulo_errado', 'ironia', 'negacao', 'misto', 'ambiguo', 'erro_do_modelo', 'outro']
resumo_anotacao = None

if os.path.exists(caminho_anotado):
    anotados = pd.read_csv(caminho_anotado)
    anotados['categoria'] = anotados['categoria'].fillna('').str.strip().str.lower()
    preenchidos = anotados[anotados['categoria'] != '']
    fora_da_lista = sorted(set(preenchidos['categoria']) - set(categorias))
    if fora_da_lista:
        print('categorias fora da lista:', fora_da_lista)

    contagem_cat = preenchidos['categoria'].value_counts().reindex(categorias, fill_value=0)
    resumo_anotacao = (contagem_cat / max(len(preenchidos), 1)).round(3).to_dict()
    print(f'{len(preenchidos)} erros anotados\n')
    print(pd.DataFrame({'erros': contagem_cat,
                        'proporção': (contagem_cat / max(len(preenchidos), 1)).map('{:.0%}'.format)}))

    fig, ax = plt.subplots(figsize=(7, 3.5))
    contagem_cat.sort_values().plot.barh(ax=ax, color='#0072B2')
    ax.set_xlabel('Número de erros')
    ax.set_title('Categorias dos erros (amostra anotada)')
    plt.tight_layout()
    plt.savefig(os.path.join(PASTA, 'categorias_de_erro.png'), dpi=150)
    plt.show()
else:
    print('erros_anotados.csv ainda não existe')

*(escrever: comentar o resultado da anotação)*

## Testando com frases novas

In [ ]:
def classificar(textos):
    textos = list(textos)
    ds = ResenhasDataset(textos, [0] * len(textos))
    probs = predict_proba(model, DataLoader(ds, batch_size=32))
    return pd.DataFrame({
        'texto': textos,
        'sentimento': [ROTULOS[int(p >= limiar)] for p in probs],
        'prob_positivo': np.round(probs, 4),
    })


exemplos = [
    'Produto excelente, chegou antes do prazo e superou minhas expectativas.',
    'Péssimo, parou de funcionar na primeira semana e ninguém resolve.',
    'O filme é lindo, mas o final deixa a desejar.',
    'Não recomendo, perda de dinheiro.',
    'Adorei o jogo, muito divertido!',
    'Não é ruim, mas também não é nada demais.',
    'Nossa, que maravilha, travou de novo bem na hora que eu mais precisava.',
]

frases = classificar(exemplos)
frases

*(escrever: comentar as previsões, principalmente nos casos de sentimento misto, negação e ironia)*